In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

import torch
import torchvision
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import time
from pathlib import Path
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version:     {torch.__version__}")
print(f"TorchVision version: {torchvision.__version__}")

# ## PyTorch Tensors
# ### Tensor Question 1

In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

b = torch.zeros(2, 3)
c = torch.ones(4)

for name, tensor in [("a", a), ("b", b), ("c", c)]:
    print(f"\n{name}:")
    print(tensor)
    print("shape:", tensor.shape)
    print("dtype:", tensor.dtype)
    print("device:", tensor.device)

# These tensors are currently on CPU unless the default device was changed.
# In training, model weights and input tensors must be on the same device,
# otherwise PyTorch cannot run the operations correctly.

# ### Tensor Question 2

In [ ]:
x = torch.tensor([1.0, 4.0, 9.0, 16.0, 25.0])

print("sqrt:", torch.sqrt(x))
print("sum:", x.sum())
print("mean:", x.mean())
print("argmax index:", x.argmax())

# In classification, argmax gives the index of the class with the highest score.

# ### Tensor Question 3

In [ ]:
a_gpu = a.to(device)
print(f"a_gpu device: {a_gpu.device}")

a_back = a_gpu.cpu()
a_numpy = a_back.numpy()

print(f"numpy type: {type(a_numpy)}")
print(f"numpy values:\n{a_numpy}")

# PyTorch requires .cpu() before .numpy() because NumPy arrays live in CPU memory,
# not GPU memory.

# ### Tensor Question 4

In [ ]:
t = torch.arange(24).float()

t_4_6 = t.reshape(4, 6)
print("Shape (4, 6):", t_4_6.shape)

t_2_3_4 = t.reshape(2, 3, 4)
print("Shape (2, 3, 4):", t_2_3_4.shape)

t_batch = t_4_6.unsqueeze(0)
print("After unsqueeze:", t_batch.shape)

# unsqueeze(0) adds a batch dimension.
# This matters because models expect inputs shaped like:
# (batch_size, channels, height, width).

# ### Tensor Question 5

In [ ]:
np_a = np.array([[1.0, 2.0], [3.0, 4.0]])
np_b = np.array([[5.0, 6.0], [7.0, 8.0]])

t_a = torch.tensor(np_a, dtype=torch.float32)
t_b = torch.tensor(np_b, dtype=torch.float32)

np_result = np_a @ np_b
torch_result = t_a @ t_b

print("NumPy result:")
print(np_result)

print("\nPyTorch result:")
print(torch_result)

print("\nDo they match?")
print(np.allclose(np_result, torch_result.numpy()))

# Matrix multiplication is one of the core operations inside neural network layers.
# A layer multiplies input values by learned weights to produce new features.

# ## Pretrained Models
# ### Model Question 1

In [ ]:
weights = ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# ResNet18 has millions of parameters, so training from scratch would require a lot of data and time.
# Using pretrained weights saves time and resources.

# ### Model Question 2

In [ ]:
print(model)

# The final layer is called "fc" and outputs 1000 values (ImageNet classes).
# The layers named layer1 to layer4 are deeper parts of the network that extract features.
# A "deep" network means it has many layers that learn increasingly complex patterns.

# ### Model Question 3

In [ ]:
model = model.to(device)
model.eval()

print("Model ready for inference.")

# .to(device) moves the model to GPU or CPU depending on device
# model.eval() sets the model to evaluation mode (disables dropout, batchnorm updates)

# ### Model Question 4

In [ ]:
preprocess = weights.transforms()
print(preprocess)

# Resize and crop prepare the image to the expected size (224x224)
# ToTensor() converts image pixels to values between 0 and 1
# Normalization adjusts pixel values based on ImageNet statistics

# ## Running Inference

In [ ]:
random.seed(42)

DATA_DIR = Path("/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test")
LABELS = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

def load_sample_image(label):
    """Load a random image file from the given class folder."""
    class_dir = DATA_DIR / label
    img_path = random.choice(list(class_dir.glob("*.jpg")))
    return Image.open(img_path).convert("RGB"), img_path.name

imagenet_classes = weights.meta["categories"]
print(f"Number of classes: {len(imagenet_classes)}")
print(f"First 5 labels: {imagenet_classes[:5]}")

# ### Inference Question 1

In [ ]:
def get_top5_predictions(model, preprocess, image, device, class_labels):
    """
    Run inference on a PIL image and return the top-5 predictions.
    Returns a list of (class_name, probability) tuples.
    """
    input_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)

    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    top_probs, top_indices = torch.topk(probabilities, 5)

    results = []
    for prob, idx in zip(top_probs, top_indices):
        class_name = class_labels[idx.item()]
        results.append((class_name, prob.item()))

    return results

img, img_name = load_sample_image("mountain")
preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)

print(f"\nTop-5 predictions for '{img_name}':")
for class_name, prob in preds:
    print(f"  {class_name:30s}  {prob:.4f}")

# The top prediction may not say "mountain" exactly because ImageNet uses labels like alp, valley, or lakeside.
# I would check whether the top-5 labels still describe something visually related to a mountain scene.

# ### Inference Question 2

In [ ]:
for label in LABELS:
    img, img_name = load_sample_image(label)
    preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)[:3]

    print(f"\n[{label}]  {img_name}")
    for class_name, prob in preds:
        print(f"  {class_name:30s}  {prob:.4f}")

# The model is usually more confident when the scene has objects or patterns close to ImageNet classes.
# It may be less confident when the scene label is broad, like sea or street.

# ### Inference Question 3

In [ ]:
img, _ = load_sample_image("forest")
input_tensor = preprocess(img).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_tensor)

probs = torch.nn.functional.softmax(logits[0], dim=0)

print(f"Logit  range: min={logits.min():.2f}, max={logits.max():.2f}")
print(f"Prob   range: min={probs.min():.6f}, max={probs.max():.4f}")
print(f"Probs sum to: {probs.sum():.6f}")
print(f"Top prediction: {imagenet_classes[probs.argmax().item()]}  ({probs.max():.4f})")

# Neural networks output logits because they are easier and more stable to work with internally.
# In production, I would use probabilities because they are easier to interpret for confidence thresholds.

# ### Inference Question 4


In [ ]:
img, img_name = load_sample_image("mountain")
preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)

class_names = [p[0] for p in preds]
probabilities = [p[1] for p in preds]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img)
axes[0].set_title(f"Image: {img_name}")
axes[0].axis("off")

axes[1].barh(class_names[::-1], probabilities[::-1])
axes[1].set_xlabel("Probability")
axes[1].set_title("Top-5 Predictions")

plt.tight_layout()
plt.savefig("outputs/warmup_inference_viz.png")
plt.show()

# For a dashboard, I would show the image, top prediction, confidence score, and top alternatives.
# I might flag predictions below 0.70 confidence for human review.

# ### Inference Question 2

In [ ]:
for label in LABELS:
    img, img_name = load_sample_image(label)
    preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)[:3]

    print(f"\n[{label}]  {img_name}")
    for class_name, prob in preds:
        print(f"  {class_name:30s}  {prob:.4f}")

# The model is more confident when the image is similar to ImageNet classes.
# It is less confident for general scenes like sea or street.

# ### Inference Question 3

In [ ]:
img, _ = load_sample_image("forest")
input_tensor = preprocess(img).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_tensor)

probs = torch.nn.functional.softmax(logits[0], dim=0)

print(f"Logit range: min={logits.min():.2f}, max={logits.max():.2f}")
print(f"Prob range: min={probs.min():.6f}, max={probs.max():.4f}")
print(f"Sum of probabilities: {probs.sum():.6f}")

# Logits are raw outputs of the model.
# Probabilities are easier to interpret for humans.